# NB14 — Re-encode images → HF `Image` type  (v2 ➜ v3)

Copies `Shanmuk4622/ai-detection-dataset-v2` into a **new** repo
`Shanmuk4622/ai-detection-dataset-v3`, converting the `image` column from raw `binary`
into the Hugging Face **`Image`** type (`struct{bytes, path}` + embedded feature metadata).
That single change makes thumbnails render in the Dataset Viewer / Data Studio.
**No pixels are altered** — the PNG bytes are copied verbatim.

## The commit / push model (what you asked for)
This notebook keeps **committing** and **pushing** strictly separate:

- **Commit = local, every shard.** As soon as a shard is converted it is written to a local
  staging folder and recorded in a local progress ledger. This is cheap, offline, and happens
  after *every* shard — so your progress is captured continuously.
- **Push = to Hugging Face, throttled.** Staged commits are pushed to HF **only** when one of
  these fires: **≥ 20 minutes** since the last push, a **major cell finishes**, the staged batch
  hits a **size cap**, or **you stop / interrupt** the run. Each push is a *single* HF commit,
  keeping you far under HF's ~128-commits/hour limit.

## Fail-proof / resumable
The **remote** ledger `_conversion_state.json` on v3 is the source of truth for resuming. If the
session dies or you press Stop, just **Run All again** — it re-reads the remote ledger and
continues from the first shard that wasn't yet *pushed*. At most the last un-pushed batch is
redone. Pressing **Stop triggers an immediate push** of the staged batch before exit.

## Kaggle setup (once)
- **Internet: ON.** GPU not needed (CPU/IO job; a dual-T4 session is fine, it just idles the GPUs).
- Add secret **`HF_TOKEN`** = a HF **write** token for `Shanmuk4622`.
- ~20 GB disk: shards are processed one at a time and deleted right after each push, so disk
  stays low regardless of the ~24 GB total dataset size.


In [ ]:
# 1) deps ---------------------------------------------------------------
import importlib.util, subprocess, sys
need = [p for m,p in [("datasets","datasets>=2.19"),
                      ("huggingface_hub","huggingface_hub>=0.24"),
                      ("pyarrow","pyarrow"), ("PIL","pillow"), ("yaml","pyyaml")]
        if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable,"-m","pip","install","-q",*need], check=True)
    print("installed:", need)
else:
    print("all deps present")

In [ ]:
# 2) config + token ----------------------------------------------------
import os, io, json, time, base64, shutil, signal, atexit, glob, traceback
import pyarrow as pa, pyarrow.parquet as pq
import yaml
from huggingface_hub import HfApi, hf_hub_download, CommitOperationAdd
from huggingface_hub.utils import HfHubHTTPError
import datasets
from datasets import Dataset, Image

SRC_REPO  = "Shanmuk4622/ai-detection-dataset-v2"
DST_REPO  = "Shanmuk4622/ai-detection-dataset-v3"
REPO_TYPE = "dataset"

# ---- push throttle (commit is always local & every-shard) ----
PUSH_EVERY_SEC   = 20 * 60      # push at least every 20 minutes
MAX_STAGE_BYTES  = 6 * 1024**3  # ...or sooner if the staged batch reaches ~6 GB
MIN_PUSH_GAP_SEC = 45           # never push more often than this
MAX_PUSH_PER_HR  = 90           # stay under HF's ~128/hr commit ceiling
PUSH_RETRIES     = 6            # retry pushes on 429 / transient errors

WORK  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/work"
STAGE = os.path.join(WORK, "stage")   # converted shards committed locally, awaiting push
DLDIR = os.path.join(WORK, "dl")      # raw v2 downloads (deleted right after convert)
LOCAL_LEDGER = os.path.join(WORK, "local_progress.json")
os.makedirs(STAGE, exist_ok=True); os.makedirs(DLDIR, exist_ok=True)

def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN","HUGGINGFACE_TOKEN","HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()

TOKEN = load_token()
api = HfApi(token=TOKEN)
print("whoami:", api.whoami()["name"], "|", SRC_REPO, "->", DST_REPO)

In [ ]:
# 3) helpers: enumerate + convert (a convert == a LOCAL commit) --------
STATE_PATH   = "_conversion_state.json"          # remote ledger (source of truth for resume)
COPY_THROUGH = ["manifest.parquet","splits.parquet","config.json",
                "validation_report.json","ai_detect_common.py"]

def ensure_dst():
    api.create_repo(DST_REPO, repo_type=REPO_TYPE, private=True, exist_ok=True)

def load_remote_state():
    try:
        p = hf_hub_download(DST_REPO, STATE_PATH, repo_type=REPO_TYPE, token=TOKEN,
                            force_download=True)
        s = json.load(open(p))
    except Exception:
        s = {}
    s.setdefault("pushed", [])      # source paths converted AND pushed to HF
    s.setdefault("meta_done", False)
    s.setdefault("readme_done", False)
    return s

def list_image_shards():
    files = api.list_repo_files(SRC_REPO, repo_type=REPO_TYPE)
    return sorted(f for f in files
                  if f.endswith(".parquet") and (f.startswith("real/") or f.startswith("data/")))

def write_local_ledger(committed, pushed):
    json.dump({"committed_not_pushed": sorted(committed), "pushed": sorted(pushed),
               "updated": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
              open(LOCAL_LEDGER, "w"), indent=2)

# binary 'image' -> HF Image struct, written with proper viewer metadata (bytes unchanged)
def convert_shard(src_path):
    local = hf_hub_download(SRC_REPO, src_path, repo_type=REPO_TYPE, token=TOKEN, local_dir=DLDIR)
    tbl   = pq.read_table(local)
    names = tbl.schema.names
    assert "image" in names, f"no image column in {src_path}: {names}"
    img_bytes = tbl.column("image").to_pylist()
    ids = tbl.column("image_id").to_pylist() if "image_id" in names else \
          [f"{i:08d}" for i in range(len(img_bytes))]
    struct = pa.array(
        [{"bytes": b, "path": (f"{iid}.png" if iid else None)} for b, iid in zip(img_bytes, ids)],
        type=pa.struct([("bytes", pa.binary()), ("path", pa.string())]))
    tbl2 = tbl.set_column(names.index("image"), "image", struct)
    ds = Dataset(tbl2).cast_column("image", Image())     # attaches Image feature + metadata
    out = os.path.join(STAGE, src_path); os.makedirs(os.path.dirname(out), exist_ok=True)
    ds.to_parquet(out)
    try: os.remove(local)                                # free disk immediately
    except OSError: pass
    return out, os.path.getsize(out)

print("convert helpers ready")

In [ ]:
# 4) Pusher: local commits accumulate; push to HF only on the triggers --
class Pusher:
    def __init__(self, state):
        self.state = state                # remote ledger dict (has 'pushed')
        self.staged = []                  # list of (src_path, local_out_path)
        self.last_push = time.time()
        self.push_times = []              # timestamps of HF pushes (rate-limit window)

    # --- LOCAL COMMIT: called after every shard convert (cheap, offline) ---
    def commit_local(self, src, out, size):
        self.staged.append((src, out))
        write_local_ledger([s for s,_ in self.staged], self.state["pushed"])
        print(f"    committed locally: {src}  ({size/1e6:.1f} MB)  "
              f"[staged {len(self.staged)}, {self.staged_bytes()/1e9:.2f} GB, "
              f"next push in {max(0,(PUSH_EVERY_SEC-(time.time()-self.last_push)))/60:.1f} min]")

    def staged_bytes(self):
        return sum(os.path.getsize(p) for _,p in self.staged if os.path.exists(p))

    def due(self):
        return self.staged and ((time.time()-self.last_push) >= PUSH_EVERY_SEC
                                 or self.staged_bytes() >= MAX_STAGE_BYTES)

    def _respect_rate_limit(self):
        now = time.time()
        self.push_times = [t for t in self.push_times if now-t <= 3600]
        if len(self.push_times) >= MAX_PUSH_PER_HR:
            wait = 3600 - (now-self.push_times[0]) + 5
            print(f"    [rate-limit] {MAX_PUSH_PER_HR} pushes/hr; sleeping {wait:.0f}s")
            time.sleep(max(wait,0))

    # --- PUSH: one HF commit for the whole staged batch + remote ledger ---
    def push(self, reason, force=False):
        if not self.staged:
            return
        if not force and (time.time()-self.last_push) < MIN_PUSH_GAP_SEC:
            return
        ops, newly = [], []
        for src, local in self.staged:
            ops.append(CommitOperationAdd(path_in_repo=src, path_or_fileobj=local)); newly.append(src)
        pushed = sorted(set(self.state["pushed"]) | set(newly))
        st = dict(self.state); st["pushed"] = pushed
        ops.append(CommitOperationAdd(path_in_repo=STATE_PATH,
                                      path_or_fileobj=json.dumps(st, indent=2).encode()))
        print(f"  >> PUSH to HF: {len(newly)} shard(s) [{reason}, {self.staged_bytes()/1e9:.2f} GB]")
        self._respect_rate_limit()
        for attempt in range(1, PUSH_RETRIES+1):
            try:
                api.create_commit(DST_REPO, repo_type=REPO_TYPE, operations=ops,
                                  commit_message=f"v3 convert: +{len(newly)} shards ({reason})")
                break
            except HfHubHTTPError as e:
                sc = getattr(getattr(e,'response',None),'status_code',None)
                back = min(60*attempt, 300)
                print(f"     push retry {attempt}/{PUSH_RETRIES} http={sc}: sleeping {back}s")
                time.sleep(back)
                if attempt == PUSH_RETRIES: raise
            except Exception as e:
                back = min(30*attempt, 180)
                print(f"     push retry {attempt}/{PUSH_RETRIES} {type(e).__name__}: sleeping {back}s")
                time.sleep(back)
                if attempt == PUSH_RETRIES: raise
        # success: update ledgers, clear stage, reclaim disk
        self.state["pushed"] = pushed
        self.push_times.append(time.time()); self.last_push = time.time()
        for _, local in self.staged:
            try: os.remove(local)
            except OSError: pass
        self.staged = []
        write_local_ledger([], self.state["pushed"])
        for d in glob.glob(os.path.join(STAGE,"*")): shutil.rmtree(d, ignore_errors=True)
        shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)

PUSHER = None
def _on_signal(signum, frame):
    print(f"\n[signal {signum}] STOP -> immediate push of staged batch ...")
    try:
        if PUSHER: PUSHER.push("interrupt", force=True)
    finally:
        raise KeyboardInterrupt
for _s in (signal.SIGINT, signal.SIGTERM):
    try: signal.signal(_s, _on_signal)
    except Exception: pass
atexit.register(lambda: PUSHER and PUSHER.push("final", force=True))
print("pusher ready")

In [ ]:
# 5) one-time: create v3, copy metadata, write card (each = a push) ----
ensure_dst()
STATE  = load_remote_state()
PUSHER = Pusher(STATE)
print("resuming; already pushed:", len(STATE["pushed"]), "shards")

# copy-through small files verbatim -> their own push (major step complete)
if not STATE["meta_done"]:
    ops = []
    for f in COPY_THROUGH:
        try:
            lp = hf_hub_download(SRC_REPO, f, repo_type=REPO_TYPE, token=TOKEN)
            ops.append(CommitOperationAdd(path_in_repo=f, path_or_fileobj=lp))
        except Exception as e:
            print("  skip", f, "->", e)
    if ops:
        STATE["meta_done"] = True
        ops.append(CommitOperationAdd(path_in_repo=STATE_PATH,
                                      path_or_fileobj=json.dumps(STATE, indent=2).encode()))
        api.create_commit(DST_REPO, repo_type=REPO_TYPE, operations=ops,
                          commit_message="v3: copy metadata files from v2")
        print("pushed metadata files")

# dataset card (README.md) with proper YAML: license/tags + per-generator configs
if not STATE["readme_done"]:
    GENS=["sd15","sdxl","flux_schnell","kandinsky22","pixart_sigma","wuerstchen"]
    configs=[{"config_name":"default","data_files":[{"split":"train","path":["real/*.parquet","data/*/*.parquet"]}]},
             {"config_name":"real","data_files":[{"split":"train","path":"real/*.parquet"}]}]
    for g in GENS: configs.append({"config_name":g,"data_files":[{"split":"train","path":f"data/{g}/*.parquet"}]})
    front={"license":"other","task_categories":["image-classification"],"language":["en"],
           "tags":["ai-generated-image-detection","synthetic-image-detection","diffusion-models","image-forensics"],
           "pretty_name":"AI-Generated Image Detection Dataset v3","size_categories":["10K<n<100K"],
           "configs":configs}
    body=base64.b64decode("IyBBSS1HZW5lcmF0ZWQgSW1hZ2UgRGV0ZWN0aW9uIERhdGFzZXQgKHYzKQoKKipQYWlyZWQgcmVhbCAvIEFJIGltYWdlcyBmb3IgdHJhaW5pbmcgYW5kIGV2YWx1YXRpbmcgQUktZ2VuZXJhdGVkLWltYWdlIGRldGVjdG9ycy4qKgoKRXZlcnkgcmVhbCBpbWFnZSBpcyBtYXRjaGVkIHdpdGggKipvbmUgQUktZ2VuZXJhdGVkIHBhcnRuZXIgcGVyIGdlbmVyYXRvcioqLCBhbmQgZWFjaApwYWlyIHNoYXJlcyBhIHNpbmdsZSwgaW1hZ2UtZ3JvdW5kZWQgY2FwdGlvbi4gQmVjYXVzZSB0aGUgcmVhbCBpbWFnZSBhbmQgYWxsIG9mIGl0cwpzeW50aGV0aWMgcGFydG5lcnMgZGVwaWN0IHRoZSAqc2FtZSBzY2VuZSogZGVzY3JpYmVkIGJ5IHRoZSAqc2FtZSBwcm9tcHQqLCBhIGRldGVjdG9yCnRyYWluZWQgaGVyZSBpcyBwdXNoZWQgdG93YXJkIHRoZSAqKnN5bnRoZXNpcyBmaW5nZXJwcmludCoqICh0ZXh0dXJlLCBmcmVxdWVuY3kgYW5kCnJlbmRlcmluZyBhcnRlZmFjdHMpIGluc3RlYWQgb2Ygc2NlbmUgY29udGVudCDigJQgdGhlIHNob3J0Y3V0IHRoYXQgaW5mbGF0ZXMgc2NvcmVzIG9uCm5haXZlbHktYnVpbHQgZGF0YXNldHMuCgo+ICoqdjMgbm90ZToqKiB2MyBpcyBpZGVudGljYWwgaW4gY29udGVudCB0byB2MiwgYnV0IHRoZSBgaW1hZ2VgIGNvbHVtbiBpcyByZS1lbmNvZGVkCj4gaW50byB0aGUgSHVnZ2luZyBGYWNlIGBJbWFnZWAgdHlwZSBzbyB0aHVtYm5haWxzIHJlbmRlciBjb3JyZWN0bHkgaW4gdGhlIERhdGFzZXQKPiBWaWV3ZXIgLyBEYXRhIFN0dWRpby4gTm8gcGl4ZWxzIHdlcmUgY2hhbmdlZCDigJQgdGhlIHVuZGVybHlpbmcgUE5HIGJ5dGVzIGFyZSBieXRlLWZvci1ieXRlCj4gdGhlIHNhbWUgYXMgdjIuCgojIyBBdCBhIGdsYW5jZQoKfCB8IHwKfC0tLXwtLS18CnwgUmVhbCBpbWFnZXMgfCAxMCwwMDAgKENPQ08gKyBJbWFnZU5ldC0xayksIGxhYmVsIGAwYCB8CnwgQUkgaW1hZ2VzIHwgNjAsMDAwIGFjcm9zcyA2IGdlbmVyYXRvcnMsIGxhYmVsIGAxYCB8CnwgUGFpcnMgfCAxMCwwMDAgKGVhY2ggcmVhbCBoYXMgNiBBSSBwYXJ0bmVycykgfAp8IFJlc29sdXRpb24gfCA1MTLDlzUxMiBSR0IgUE5HIHwKfCBQcmVwcm9jZXNzaW5nIHwgSWRlbnRpY2FsIGNhbm9uaWNhbCBwaXBlbGluZSBmb3IgcmVhbCAqKmFuZCoqIEFJIChgcGlwZWxpbmVfdmVyc2lvbj0xLjJgKSB8CnwgTWFzdGVyIHNlZWQgfCBgNDJgIOKAlCBjb250cm9scyBnZW5lcmF0aW9uIHNlZWRzIGFuZCBzcGxpdCBhc3NpZ25tZW50IHwKfCBTcGxpdCB8IDcwIC8gMTUgLyAxNSBwYWlyLWxldmVsIHRyYWluIC8gdmFsIC8gdGVzdCB8CnwgTGljZW5zZSB8IE5vbi1jb21tZXJjaWFsIHJlc2VhcmNoIHVzZSAobW9zdC1yZXN0cmljdGl2ZSBpbmhlcml0ZWQgdGVybSkgfAoKIyMgR2VuZXJhdG9ycwoKfCBrZXkgfCBtb2RlbCBpZCB8IG5hdGl2ZSByZXMgfCBzdGVwcyB8IGd1aWRhbmNlIHwKfC0tLS0tfC0tLS0tLS0tLS18LS0tLS0tLS0tLS18LS0tLS0tLXwtLS0tLS0tLS0tfAp8IGBzZDE1YCB8IGBzdGFibGUtZGlmZnVzaW9uLXYxLTUvc3RhYmxlLWRpZmZ1c2lvbi12MS01YCB8IDUxMiB8IDIwIHwgNy4wIHwKfCBgc2R4bGAgfCBgc3RhYmlsaXR5YWkvc3RhYmxlLWRpZmZ1c2lvbi14bC1iYXNlLTEuMGAgfCAxMDI0IHwgOCB8IDcuMCB8CnwgYGZsdXhfc2NobmVsbGAgfCBgYmxhY2stZm9yZXN0LWxhYnMvRkxVWC4xLXNjaG5lbGxgIHwgMTAyNCB8IDQgfCAwLjAgfAp8IGBrYW5kaW5za3kyMmAgfCBga2FuZGluc2t5LWNvbW11bml0eS9rYW5kaW5za3ktMi0yLWRlY29kZXJgIHwgNzY4IHwgMjUgfCA0LjAgfAp8IGBwaXhhcnRfc2lnbWFgIHwgYFBpeEFydC1hbHBoYS9QaXhBcnQtU2lnbWEtWEwtMi0xMDI0LU1TYCB8IDEwMjQgfCAyMCB8IDQuNSB8CnwgYHd1ZXJzdGNoZW5gIHwgYHdhcnAtYWkvd3VlcnN0Y2hlbmAgfCAxMDI0IHwgWzIwLCAxMF0gfCBbNC4wLCAwLjBdIHwKCiMjIEhvdyB0aGUgcGFpcnMgYXJlIGJ1aWx0CgpSZWFsIGltYWdlcyB3ZXJlIGNhcHRpb25lZCB3aXRoICoqU2FsZXNmb3JjZS9ibGlwMi1vcHQtMi43YioqLCBjbGVhbmVkIGFuZCBjYXBwZWQgdG8KNzUgQ0xJUCB0b2tlbnMsIHRoZW4gcmV1c2VkICoqYnl0ZS1pZGVudGljYWxseSoqIGJ5IGFsbCBzaXggZ2VuZXJhdG9ycy4gVGhlIGNvbHVtbgpgc291cmNlX3JlYWxfaWRgIGxpbmtzIGV2ZXJ5IEFJIGltYWdlIGJhY2sgdG8gaXRzIHJlYWwgcGFydG5lcgooYHNvdXJjZV9yZWFsX2lkID09IGltYWdlX2lkYCBmb3IgcmVhbCByb3dzKS4KCiMjIENhbm9uaWNhbCBwcmVwcm9jZXNzIChsZWFrLWZyZWUpCgpCb3RoIHJlYWwgYW5kIEFJIGltYWdlcyBwYXNzIHRocm91Z2ggdGhlICoqc2FtZSoqIHBpcGVsaW5lOgoKMS4gRVhJRi10cmFuc3Bvc2Ug4oaSIGNvbnZlcnQgdG8gUkdCIOKGkiBjZW50ZXItY3JvcCB0byBzcXVhcmUg4oaSIExhbmN6b3MgcmVzaXplIHRvIDUxMi4KMi4gSlBFRy1lcXVhbGlzZSAocXVhbGl0eSA5NSwgNDo0OjQpIOKGkiBzYXZlIGFzIFBORyAoY29tcHJlc3MgbGV2ZWwgNiwgbm8gRVhJRi9JQ0MpLgoKQXBwbHlpbmcgYW4gaWRlbnRpY2FsIHRyYW5zZm9ybSB0byBib3RoIGNsYXNzZXMgcmVtb3ZlcyB0aGUgcmVzb2x1dGlvbiwgY29sb3VyLXNwYWNlCmFuZCBKUEVHLWFydGVmYWN0IHNob3J0Y3V0cyB0aGF0IHdvdWxkIG90aGVyd2lzZSBsZXQgYSBtb2RlbCAiY2hlYXQiIGluc3RlYWQgb2YgbGVhcm5pbmcKdGhlIGdlbmVyYXRpdmUgZmluZ2VycHJpbnQuCgojIyBTcGxpdHMKCkRldGVybWluaXN0aWMgKipwYWlyLWxldmVsKiogc3BsaXQga2V5ZWQgb24gYHNvdXJjZV9yZWFsX2lkYCAoc2VlZCBgNDJgKTogYSByZWFsIGltYWdlCmFuZCBhbGwgc2l4IG9mIGl0cyBBSSBwYXJ0bmVycyBhbHdheXMgbGFuZCBpbiB0aGUgc2FtZSBmb2xkLCBzbyB0aGVyZSBpcyAqKm5vIHNjZW5lCmxlYWthZ2UqKiBiZXR3ZWVuIHRyYWluIC8gdmFsIC8gdGVzdC4KCnwgU3BsaXQgfCBJbWFnZXMgfAp8LS0tLS0tLXwtLS0tLS0tLXwKfCB0cmFpbiB8IDQ5LDM5MiB8CnwgdmFsICAgfCAxMCwxMjIgfAp8IHRlc3QgIHwgMTAsNDg2IHwKClRoZSByb3ctbGV2ZWwgc3BsaXQgbGFiZWwgbGl2ZXMgaW4gYG1hbmlmZXN0LnBhcnF1ZXRgIChhbmQgYHNwbGl0cy5wYXJxdWV0YCBrZXllZCBieQpgc291cmNlX3JlYWxfaWRgKS4KCiMjIERhdGFzZXQgc3RydWN0dXJlCgpgYGAKYWktZGV0ZWN0aW9uLWRhdGFzZXQtdjMvCuKUnOKUgOKUgCByZWFsLyAgICAgICAgICAgICAgICAgICAgICAgIyAxMGsgcmVhbCBpbWFnZXMsIGxhYmVsIDAgIChyZWFsLSoucGFycXVldCkK4pSc4pSA4pSAIGRhdGEvCuKUgiAgIOKUnOKUgOKUgCBzZDE1LyAgICAgICAgICAgICAgICAgICAjIDEwayBBSSBpbWFnZXMgcGVyIGdlbmVyYXRvciwgbGFiZWwgMQrilIIgICDilJzilIDilIAgc2R4bC8K4pSCICAg4pSc4pSA4pSAIGZsdXhfc2NobmVsbC8K4pSCICAg4pSc4pSA4pSAIGthbmRpbnNreTIyLwrilIIgICDilJzilIDilIAgcGl4YXJ0X3NpZ21hLwrilIIgICDilJTilIDilIAgd3VlcnN0Y2hlbi8K4pSc4pSA4pSAIG1hbmlmZXN0LnBhcnF1ZXQgICAgICAgICAgICAjIGZ1bGwgaW5kZXg6IGlkLCBwYWlyaW5nIGtleSwgbGFiZWwsIGdlbmVyYXRvciwgc3BsaXQgKG5vIGJ5dGVzKQrilJzilIDilIAgc3BsaXRzLnBhcnF1ZXQgICAgICAgICAgICAgICMgc291cmNlX3JlYWxfaWQgLT4gc3BsaXQK4pSc4pSA4pSAIGNvbmZpZy5qc29uICAgICAgICAgICAgICAgICAjIGJ1aWxkIGNvbmZpZ3VyYXRpb24gKHNlZWRzLCBtb2RlbHMsIHBhcmFtcykK4pSc4pSA4pSAIHZhbGlkYXRpb25fcmVwb3J0Lmpzb24gICAgICAjIGxlYWstYXVkaXQgLyBzbmlmZi10ZXN0IHJlc3VsdHMK4pSU4pSA4pSAIGFpX2RldGVjdF9jb21tb24ucHkgICAgICAgICAjIHNoYXJlZCBoZWxwZXIgbGlicmFyeSB1c2VkIGJ5IHRoZSBidWlsZCBub3RlYm9va3MKYGBgCgpUaGUgRGF0YXNldCBWaWV3ZXIgZXhwb3NlcyBvbmUgKipgZGVmYXVsdGAqKiBjb25maWcgKHJlYWwgKyBhbGwgZ2VuZXJhdG9ycyBjb21iaW5lZCkgcGx1cwpvbmUgY29uZmlnICoqcGVyIGdlbmVyYXRvcioqIGFuZCBhICoqYHJlYWxgKiogY29uZmlnLCBzbyB5b3UgY2FuIGJyb3dzZSBlYWNoIHNvdXJjZSBvbiBpdHMKb3duLgoKIyMgVXNhZ2UKCiMjIyBXaXRoIGBkYXRhc2V0c2AgKHN0cmVhbWluZywgdmlld2VyLW5hdGl2ZSkKCmBgYHB5dGhvbgpmcm9tIGRhdGFzZXRzIGltcG9ydCBsb2FkX2RhdGFzZXQKCiMgY29tYmluZWQgcmVhbCArIEFJCmRzID0gbG9hZF9kYXRhc2V0KCJTaGFubXVrNDYyMi9haS1kZXRlY3Rpb24tZGF0YXNldC12MyIsIHNwbGl0PSJ0cmFpbiIsIHN0cmVhbWluZz1UcnVlKQpleCA9IG5leHQoaXRlcihkcykpCmV4WyJpbWFnZSJdICAgICAgIyBQSUwuSW1hZ2UgKGRlY29kZWQgYXV0b21hdGljYWxseSkKZXhbImxhYmVsIl0gICAgICAjIDAgPSByZWFsLCAxID0gQUkKCiMganVzdCBvbmUgZ2VuZXJhdG9yCmZsdXggPSBsb2FkX2RhdGFzZXQoIlNoYW5tdWs0NjIyL2FpLWRldGVjdGlvbi1kYXRhc2V0LXYzIiwgImZsdXhfc2NobmVsbCIsIHNwbGl0PSJ0cmFpbiIpCmBgYAoKIyMjIFdpdGggYHB5YXJyb3dgIChyYXcgYnl0ZXMsIG5vIGRlY29kZSkKCmBgYHB5dGhvbgpmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgaGZfaHViX2Rvd25sb2FkCmltcG9ydCBweWFycm93LnBhcnF1ZXQgYXMgcHEKZnJvbSBQSUwgaW1wb3J0IEltYWdlCmltcG9ydCBpbwoKc2hhcmQgPSBoZl9odWJfZG93bmxvYWQoIlNoYW5tdWs0NjIyL2FpLWRldGVjdGlvbi1kYXRhc2V0LXYzIiwKICAgICAgICAgICAgICAgICAgICAgICAgImRhdGEvZmx1eF9zY2huZWxsL2ZsdXhfc2NobmVsbC0zMDFmZDkyZS0wMDAwMi5wYXJxdWV0IiwKICAgICAgICAgICAgICAgICAgICAgICAgcmVwb190eXBlPSJkYXRhc2V0IikKdGJsID0gcHEucmVhZF90YWJsZShzaGFyZCkKaW1nID0gSW1hZ2Uub3Blbihpby5CeXRlc0lPKHRibFsiaW1hZ2UiXVswXVsiYnl0ZXMiXS5hc19weSgpKSkgICAjIHYzOiBpbWFnZSBpcyBhIHN0cnVjdApgYGAKCj4gKipNaWdyYXRpbmcgZnJvbSB2Mj8qKiBJbiB2MiB0aGUgYGltYWdlYCBjb2x1bW4gd2FzIHJhdyBgYmluYXJ5YAo+IChgdGJsWyJpbWFnZSJdWzBdLmFzX3B5KClgKS4gSW4gdjMgaXQgaXMgYSBgc3RydWN0e2J5dGVzLCBwYXRofWAsIHNvIHJlYWQKPiBgdGJsWyJpbWFnZSJdWzBdWyJieXRlcyJdLmFzX3B5KClgLgoKIyMgU2NoZW1hCgp8IGNvbHVtbiB8IHR5cGUgfCBkZXNjcmlwdGlvbiB8CnwtLS0tLS0tLXwtLS0tLS18LS0tLS0tLS0tLS0tLXwKfCBgaW1hZ2VgIHwgSW1hZ2UgYHN0cnVjdHtieXRlcywgcGF0aH1gIHwgY2Fub25pY2FsIDUxMsOXNTEyIFJHQiBQTkcgfAp8IGBpbWFnZV9pZGAgfCBzdHJpbmcgfCBnbG9iYWxseSB1bmlxdWUsIGUuZy4gYHJlYWxfMDAwMDAxYCAvIGBzZHhsXzAwMDAwMWAgfAp8IGBzb3VyY2VfcmVhbF9pZGAgfCBzdHJpbmcgfCBwYWlyaW5nIGtleTsgZXF1YWxzIGBpbWFnZV9pZGAgZm9yIHJlYWwgcm93cyB8CnwgYGxhYmVsYCB8IGludDggfCAwID0gcmVhbCwgMSA9IEFJIHwKfCBgZ2VuZXJhdG9yYCB8IHN0cmluZyB8IGByZWFsYCwgYHNkMTVgLCBgc2R4bGAsIGBmbHV4X3NjaG5lbGxgLCBga2FuZGluc2t5MjJgLCBgcGl4YXJ0X3NpZ21hYCwgYHd1ZXJzdGNoZW5gIHwKfCBgc291cmNlX2RhdGFzZXRgIHwgc3RyaW5nIHwgYGNvY29gLCBgaW1hZ2VuZXRgLCBvciBgPGdlbmVyYXRvcj5gIHwKfCBgcHJvbXB0YCB8IHN0cmluZyB8IGNhcHRpb24gdXNlZCAoQUkgcm93cyk7IG51bGwgZm9yIHJlYWwgfAp8IGB3aWR0aGAgLyBgaGVpZ2h0YCB8IGludDE2IHwgYWx3YXlzIDUxMiB8CnwgYG9yaWdfd2lkdGhgIC8gYG9yaWdfaGVpZ2h0YCB8IGludDMyIHwgcHJvdmVuYW5jZSBvbmx5IOKAlCAqKm5ldmVyIHVzZSBhcyBhIHRyYWluaW5nIGZlYXR1cmUqKiB8CnwgYGdlbl9tb2RlbF9pZGAgfCBzdHJpbmcgfCBIRiBtb2RlbCBpZCB8CnwgYGdlbl9zdGVwc2AgLyBgZ2VuX2d1aWRhbmNlYCAvIGBnZW5fbmF0aXZlX3Jlc2AgfCBpbnQxNiAvIGZsb2F0MzIgLyBpbnQxNiB8IGdlbmVyYXRpb24gcGFyYW1zIHwKfCBgc2VlZGAgfCBpbnQ2NCB8IGBoYXNoKG1hc3Rlcl9zZWVkLCBnZW5lcmF0b3IsIHNvdXJjZV9yZWFsX2lkKWAgfAp8IGBjYXB0aW9uX21vZGVsYCB8IHN0cmluZyB8IGNhcHRpb25lciB1c2VkIHwKfCBgcGlwZWxpbmVfdmVyc2lvbmAgfCBzdHJpbmcgfCBgMS4yYCB8CnwgYHNoYTI1NmAgfCBzdHJpbmcgfCBoZXggZGlnZXN0IG9mIHRoZSBQTkcgYnl0ZXMgfAp8IGBjcmVhdGVkX3V0Y2AgfCBzdHJpbmcgfCBJU08tODYwMSBVVEMgdGltZXN0YW1wIHwKCiMjIExlYWsgYXVkaXQKCkEgInNuaWZmIHRlc3QiIHRyYWlucyBhIGxpZ2h0d2VpZ2h0IGNsYXNzaWZpZXIgb24gZWFjaCBnZW5lcmF0b3IgdnMuIHJlYWwgdXNpbmcgb25seQpsb3ctbGV2ZWwgc3RhdGlzdGljcyBhbmQgY2hlY2tzIHRoYXQgc2VwYXJhYmlsaXR5IHN0YXlzIGluIGEgaGVhbHRoeSBiYW5kCihgPDAuODVgIGhlYWx0aHksIGAwLjg14oCTMC45NWAgaW52ZXN0aWdhdGUsIGA+MC45NWAgbGVhaykuIExhdGVzdCByZXN1bHRzOgoKfCBnZW5lcmF0b3IgfCBzY29yZSB8CnwtLS0tLS0tLS0tLXwtLS0tLS0tfAp8IHNkMTUgfCAwLjkxMCB8Cnwgc2R4bCB8IDAuOTczIHwKfCBmbHV4X3NjaG5lbGwgfCAwLjkxNyB8Cnwga2FuZGluc2t5MjIgfCAwLjk1OCB8CnwgcGl4YXJ0X3NpZ21hIHwgMC45NDMgfAp8IHd1ZXJzdGNoZW4gfCAwLjg0MyB8CnwgKipvdmVyYWxsKiogfCAqKjAuODYwKiogfAoKRnVsbCByZXN1bHRzIGluIGB2YWxpZGF0aW9uX3JlcG9ydC5qc29uYC4KCiMjIFByb3ZlbmFuY2UsIGxpY2Vuc2UgYW5kIGludGVuZGVkIHVzZQoKLSAqKkltYWdlTmV0KiogY29udGVudCBpcyAqKm5vbi1jb21tZXJjaWFsIHJlc2VhcmNoIG9ubHkqKiAoSUxTVlJDIHRlcm1zKS4KLSAqKkNPQ08qKiBpbWFnZXMgYXJlIEZsaWNrci1zb3VyY2VkIChDcmVhdGl2ZSBDb21tb25zKS4KLSBBSSBpbWFnZXMgYXJlIHN5bnRoZXRpYyBvdXRwdXRzIG9mIHRoZSBtb2RlbHMgbGlzdGVkIGFib3ZlLCBlYWNoIHVuZGVyIGl0cyBvd24gbGljZW5zZS4KLSBUaGlzIGRhdGFzZXQgaW5oZXJpdHMgdGhlICoqbW9zdCByZXN0cmljdGl2ZSoqIGFwcGxpY2FibGUgdGVybTogKipub24tY29tbWVyY2lhbAogIHJlc2VhcmNoIHVzZSoqLiAqTm90IGxlZ2FsIGFkdmljZS4qCi0gSW50ZW5kZWQgZm9yIHRyYWluaW5nIC8gZXZhbHVhdGluZyBBSS1pbWFnZSBkZXRlY3RvcnMuCgojIyBMaW1pdGF0aW9ucyBhbmQgY2F2ZWF0cwoKLSBUaGUgY29yZSBzZXQgaXMgKip0ZXh0LXRvLWltYWdlIG9ubHkqKiAobm8gaW1nMmltZyAvIHJlZmVyZW5jZSBjb25kaXRpb25pbmcpOyBkZXRlY3RvcnMKICB0cmFpbmVkIGhlcmUgdGFyZ2V0IHRoZSB0ZXh0LXRvLWltYWdlIHRocmVhdCBtb2RlbC4KLSBDYXB0aW9uIHF1YWxpdHkgKEJMSVAtMiBjb25jaXNlIHNlbnRlbmNlcykgYm91bmRzIHByb21wdCBkaXZlcnNpdHkuCi0gRkxVWC4xLXNjaG5lbGwgYW5kIFfDvHJzdGNoZW4gcnVuIHdpdGggQ1BVIG9mZmxvYWQgb24gVDQ7IGdlbmVyYXRpb24gY29uZGl0aW9ucyBhcmUKICBvdGhlcndpc2UgY29uc2lzdGVudCB3aXRoIHN0YW5kYXJkIGluZmVyZW5jZSBzZXR0aW5ncy4KLSBQZXItbW9kZWwgc3RlcCBjb3VudHMgd2VyZSByZWR1Y2VkIGZvciBzcGVlZCAoU0RYTCA4IHN0ZXBzLCBTRCAxLjUgMjAgc3RlcHMpLCB3aGljaCBpcwogIHJlcHJlc2VudGF0aXZlIG9mIHJlYWwtd29ybGQgZmFzdCBpbmZlcmVuY2UuCgojIyBDaXRhdGlvbgoKYGBgYmlidGV4CkBtaXNje2FpZGV0ZWN0aW9uX2RhdGFzZXRfdjMsCiAgdGl0bGUgID0ge0FJLUdlbmVyYXRlZCBJbWFnZSBEZXRlY3Rpb24gRGF0YXNldCAodjMpfSwKICBhdXRob3IgPSB7U2hhbm11azQ2MjJ9LAogIHllYXIgICA9IHsyMDI1fSwKICBob3dwdWJsaXNoZWQgPSB7XHVybHtodHRwczovL2h1Z2dpbmdmYWNlLmNvL2RhdGFzZXRzL1NoYW5tdWs0NjIyL2FpLWRldGVjdGlvbi1kYXRhc2V0LXYzfX0KfQpgYGAK").decode()
    readme="---\n"+yaml.safe_dump(front,sort_keys=False,allow_unicode=True)+"---\n\n"+body
    STATE["readme_done"]=True
    api.create_commit(DST_REPO, repo_type=REPO_TYPE, commit_message="v3: dataset card",
        operations=[CommitOperationAdd("README.md", readme.encode()),
                    CommitOperationAdd(STATE_PATH, json.dumps(STATE,indent=2).encode())])
    print("pushed dataset card (", len(readme), "chars )")

In [ ]:
# 6) MAIN LOOP — convert (commit local) ; push on the triggers ---------
all_shards = list_image_shards()
todo = [s for s in all_shards if s not in set(STATE["pushed"])]
print(f"image shards: {len(all_shards)} total | {len(todo)} remaining")

PUSHER.last_push = time.time()
done_run, failed = 0, []
try:
    for i, src in enumerate(todo, 1):
        t0 = time.time()
        try:
            out, sz = convert_shard(src)           # heavy step
            PUSHER.commit_local(src, out, sz)      # <-- LOCAL COMMIT (every shard)
            done_run += 1
            print(f"[{i}/{len(todo)}] converted {src}  ({time.time()-t0:.1f}s)")
        except Exception as e:
            failed.append(src)
            print(f"[{i}/{len(todo)}] FAILED {src}: {type(e).__name__}: {e}")
            traceback.print_exc()
        if PUSHER.due():                           # 20-min / size-cap trigger
            PUSHER.push("interval")
    PUSHER.push("final", force=True)               # major cell complete -> push remainder
except KeyboardInterrupt:
    print("interrupted — staged batch already pushed by the signal handler.")
    raise
print(f"\nconverted this run: {done_run} | failed: {len(failed)}")
if failed: print("failed (retried next run):", failed[:20])

In [ ]:
# 7) verify + finish ---------------------------------------------------
STATE = load_remote_state()
all_shards = list_image_shards()
remaining = [s for s in all_shards if s not in set(STATE["pushed"])]
print(f"pushed: {len(STATE['pushed'])}/{len(all_shards)} | remaining: {len(remaining)}")
if not remaining:
    probe = next(s for s in all_shards if s.startswith("data/"))
    lp = hf_hub_download(DST_REPO, probe, repo_type=REPO_TYPE, token=TOKEN, force_download=True)
    it = pq.read_schema(lp).field("image").type
    ok = pa.types.is_struct(it) and {f.name for f in it} >= {"bytes","path"}
    print("image column type:", it, "| struct(bytes,path):", ok)
    print("\nALL SHARDS CONVERTED + PUSHED ✅")
    print("Make v3 public when ready:")
    print("  HfApi(token=TOKEN).update_repo_settings('%s', repo_type='dataset', private=False)" % DST_REPO)
    print("Viewer: https://huggingface.co/datasets/%s" % DST_REPO)
else:
    print("Not finished — Run All again to resume from the remote ledger.")